# TSTR study (ESC-50): do calibrated metrics predict how useful synthetic data is?

**The question (thesis claim 3).** For each synthetic set: how far is it from real audio (Frechet
distance divided by the real-vs-real floor, the evaluation method used so far), and how well does
a classifier trained only on it do on real test clips (TSTR)? If the distance predicts the TSTR
accuracy, the metric can be used to choose synthetic data without training anything on it.

**Pre-registered design** (`claude/tstr-study-design.md`, fixed Oct 17; Amendment 1 on Oct 27,
before any of the new sets existed):

* **15 synthetic sets.** A-64, A-128 and C, each at EMA 5k and 10k steps and guidance 1.0 and
  3.0 (12 sets), plus B raw, B codec and B latent (10k, guidance 1.0). 800 clips each, 16 per class.
* **Metrics against fold 4** (the validation fold): every set as audio, embedded with CLAP and
  PANNs, Frechet ratio to the real-vs-real floor. *Primary predictors: the two Frechet ratios.*
  Secondary, exploratory: MMD, the mean and covariance terms, the stretch log-volume, teacher
  accuracy on synthetic (TRTS), learnability.
* **TSTR on fold 5**, with the classifier recipe the pilot froze (read from the pilot's results
  file, not re-chosen), 3 seeds per set. Reported relative to Real-matched (800 real crops,
  16 per class).
* **Primary statistic:** Spearman rank correlation between each primary predictor and relative
  TSTR accuracy, with a 95% bootstrap interval that resamples **paths** (6 paths), because sets of
  one path are not independent. Also reported within factors: across paths at a fixed checkpoint and
  guidance, across guidance within a path, across checkpoints within a path.
* Differences smaller than the seed-to-seed spread are not reported as effects.

A Frechet ratio is "lower is better", so a metric that predicts usefulness gives a **negative**
correlation.

**Resumable.** Griffin-Lim audio, embeddings and every TSTR condition are saved to Drive as they
finish; rerunning after a disconnect continues where it stopped. Runtime on a T4: about 1.5 hours
in total (most of it the 51 classifier trainings), splittable across sessions.

## 1. Setup

In [ ]:
!pip install -q diffusers accelerate soundfile panns_inference "transformers>=4.45"
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, csv, json, glob, time, subprocess
REPO_DIR = "/content/thesis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/1jvaldez1/audio-diffusion-synth.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, REPO_DIR)
import common as C
import eval_metrics as M
import tstr as T

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchaudio
from pathlib import Path
from scipy.stats import spearmanr

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
commit = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"]).decode().strip()

## 2. Config

The set list, predictors and statistics are the pre-registered ones. `LEARNABILITY` holds the
path-level relative v-loss measured on Oct 16-17 (training clips, 10k steps), used only as an
exploratory, path-level predictor.

In [ ]:
QUICK_TEST = False          # True: tiny run end to end, numbers meaningless

cfg = C.Config()
cfg128 = C.Config(n_mels=128)
ROOT = cfg.out_root
S, STUDY = f"{ROOT}/samples", f"{ROOT}/samples/study" + ("_quick" if QUICK_TEST else "")
OUT_DIR = Path(f"{ROOT}/tstr/full" + ("_quick" if QUICK_TEST else ""))
OUT_DIR.mkdir(parents=True, exist_ok=True)

FRONTENDS = {                                   # path -> (front-end file, config, class)
    "A64":  (f"{ROOT}/frontend.json",        cfg,    C.MelFrontEnd),
    "A128": (f"{ROOT}/frontend_128mel.json", cfg128, C.MelFrontEnd),
    "C":    (f"{ROOT}/frontend_linear.json", cfg,    C.LinearFrontEnd),
}
EXISTING = {"A64": f"{S}/path_a.pt", "A128": f"{S}/path_a_128mel.pt", "C": f"{S}/path_c_linear.pt"}
SETS = {}                                       # name -> dict(file, kind, path, step, guidance)
for p in ("A64", "A128", "C"):
    for step in (5000, 10000):
        for g in (1.0, 3.0):
            f = EXISTING[p] if (step, g) == (10000, 1.0) else f"{STUDY}/{p}_ema{step // 1000}k_g{g:g}.pt"
            SETS[f"{p}_{step // 1000}k_g{g:g}"] = dict(file=f, kind="linear" if p == "C" else "mel", path=p, step=step, guidance=g)
for name, f in (("B_raw", "path_b_wave.pt"), ("B_codec", "path_b_codec_wave.pt"), ("B_latent", "path_b_latent_wave_k96.pt")):
    SETS[name] = dict(file=f"{S}/{f}", kind="waveform", path=name, step=10000, guidance=1.0)

LEARNABILITY = {"A64": 0.105, "A128": 0.121, "C": 0.095, "B_raw": 0.401, "B_codec": 0.93, "B_latent": 0.867}

GL_ITERS, MAX_DIM, N_SPLITS, N_PERM = 64, 32, 30, 1000
EMBEDDERS = ("clap", "panns")
N_SEEDS, N_BOOT, TEST_CROPS = 3, 2000, 3

# the classifier recipe frozen by the pilot
pilots = sorted(glob.glob(f"{ROOT}/tstr/pilot/pilot_*.json"), key=os.path.getmtime)
assert pilots, "no pilot results found: run tstr_pilot_esc50.ipynb (full mode) first"
pilot = json.load(open(pilots[-1]))
assert not pilot["quick_test"], f"{pilots[-1]} is a quick-test result"
RECIPE = pilot["recipe"]
print(f"frozen recipe from {Path(pilots[-1]).name}: {RECIPE}")

if QUICK_TEST:
    GL_ITERS, N_SPLITS, N_PERM, N_SEEDS, N_BOOT = 8, 6, 50, 2, 200
    RECIPE = {**RECIPE, "steps": 20}
missing = [n for n, s in SETS.items() if not os.path.exists(s["file"])]
for n in missing:
    print(f"MISSING {n}: {SETS[n]['file']}  (run tstr_sample_sets.ipynb)")
SETS = {n: s for n, s in SETS.items() if n not in missing}
print(f"{len(SETS)} sets present")

## 3. Data

* **Metric reference: fold 4**, two evenly spaced valid crops per clip (800 crops from 400 clips),
  grouped by source recording for the real-vs-real splits. Fold 5 is never used for a metric.
* **TSTR**: the same Real-matched crops as the pilot (same selection seed) and fold-5 test crops
  (3 per clip, probabilities averaged per clip).

In [ ]:
data = C.load_esc50(cfg)
train_split, val_split, test_split = C.split_by_fold(data, cfg)
class_names, n_classes = data["class_names"], len(data["class_names"])
meta_csv = Path(cfg.data_root) / "ESC-50-master" / "meta" / "esc50.csv"
if meta_csv.exists():
    with open(meta_csv) as f:
        src = np.array([r["src_file"] for r in csv.DictReader(f)])
    src_val, src_test = src[data["folds"].numpy() == cfg.val_fold], src[data["folds"].numpy() == cfg.heldout_fold]
else:
    print("ESC-50 metadata not found: grouping by clip instead of by source recording")
    src_val, src_test = np.arange(len(val_split["labels"])), np.arange(len(test_split["labels"]))
del data

ref_wave, ref_y, ref_clip = C.reference_crops(val_split, cfg, k=2, return_clip_index=True)
ref_groups = src_val[ref_clip.numpy()]

r = np.random.default_rng(cfg.seed)                    # identical to the pilot's Real-matched selection
ytr_all = train_split["labels"].numpy()
matched_idx = torch.from_numpy(np.sort(np.concatenate([r.choice(np.flatnonzero(ytr_all == c), 16, replace=False)
                                                       for c in range(n_classes)])))
matched_wave, matched_y = T.middle_crop(train_split, matched_idx, cfg), train_split["labels"][matched_idx]
test_wave, test_y, test_clip = C.reference_crops(test_split, cfg, k=TEST_CROPS, return_clip_index=True)
test_clip_y = test_split["labels"].numpy()
print(f"metric reference (fold 4): {len(ref_wave)} crops, {len(np.unique(ref_groups))} recordings | "
      f"Real-matched {len(matched_y)} | test (fold 5) {len(test_clip_y)} clips")

## 4. Every set to audio

Spectrogram sets through Griffin-Lim with their own path's front end (mel for A-64 and A-128,
linear for C); waveform sets as they are. A provenance check stops on any set that does not hold
800 clips, 16 per class, made with guidance and checkpoint as listed. Cached per set.

In [ ]:
_fes = {}
def frontend_of(path):
    if path not in _fes:
        f, c, cls = FRONTENDS[path]
        _fes[path] = cls.load(f, c, device=device)
    return _fes[path]

def cached(key, source_file, fn):
    p = OUT_DIR / f"{key}.pt"
    stamp = os.path.getmtime(source_file) if source_file else 0.0
    if p.exists():
        d = torch.load(p)
        if d["stamp"] == stamp:
            return d["value"]
    v = fn(); torch.save({"value": v, "stamp": stamp}, p); return v

waves, labels = {}, {}
for name, s in SETS.items():
    d = C.load_samples(s["file"])
    counts = torch.bincount(d["labels"], minlength=n_classes)
    want = 1 if QUICK_TEST and s["file"].startswith(STUDY) else 16
    assert (counts == want).all() or (QUICK_TEST and counts.min() > 0), f"{name}: class counts {counts.min().item()}-{counts.max().item()}"
    assert d["meta"].get("guidance", 1.0) == s["guidance"], f"{name}: guidance {d['meta'].get('guidance')} in the file"
    if "ema_step" in d["meta"]:
        assert d["meta"]["ema_step"] == s["step"], f"{name}: checkpoint {d['meta']['ema_step']} in the file"
    if s["kind"] == "waveform":
        w = d["samples"].float().clamp(-1, 1)
    else:
        w = cached(f"wave_{name}_gl{GL_ITERS}", s["file"],
                   lambda d=d, s=s: C.spec_to_wave(d["samples"], frontend_of(s["path"]), n_iter=GL_ITERS, seed=cfg.seed))
    assert w.shape[-1] == cfg.clip_samples, f"{name}: {w.shape[-1]} samples per clip"
    waves[name], labels[name] = w, d["labels"].long()
    print(f"{name:<14}{tuple(w.shape)}  commit {d['meta'].get('commit', '-')}")

## 5. Distribution metrics (fold 4)

CLAP and PANNs embeddings, then one `M.evaluate` call per embedder with all sets as candidates,
so every set is compared with the same real reference and the same floors.

In [ ]:
def load_panns():
    from panns_inference import AudioTagging
    tagger = AudioTagging(checkpoint_path=None, device=device)
    rs = torchaudio.transforms.Resample(cfg.sample_rate, 32000)
    return lambda w: tagger.inference(rs(w[:, 0]).numpy())[1]

def load_clap():
    from transformers import ClapModel, ClapFeatureExtractor
    model = ClapModel.from_pretrained("laion/clap-htsat-unfused").to(device).eval()
    fx = ClapFeatureExtractor.from_pretrained("laion/clap-htsat-unfused")
    rs = torchaudio.transforms.Resample(cfg.sample_rate, 48000)
    @torch.no_grad()
    def embed(w):
        audio = [a.numpy() for a in rs(w[:, 0])]
        inputs = {k: v.to(device) for k, v in fx(audio, sampling_rate=48000, truncation="rand_trunc", padding="repeatpad", return_tensors="pt").items()}
        out = model.get_audio_features(**inputs)
        return (out.pooler_output if hasattr(out, "pooler_output") else out).cpu().numpy()
    return embed

LOADERS = {"panns": load_panns, "clap": load_clap}

def embed_all(e, batch=32):
    fn, out = None, {}
    for name, w in {"real": ref_wave, **waves}.items():
        p = OUT_DIR / f"emb_{e}_{name}.npy"
        stamp = os.path.getmtime(SETS[name]["file"]) if name in SETS else 0.0
        st = p.with_suffix(".stamp")
        if p.exists() and st.exists() and float(st.read_text()) == stamp:
            out[name] = np.load(p); continue
        fn = fn or LOADERS[e]()
        out[name] = np.concatenate([fn(w[i:i + batch]) for i in range(0, len(w), batch)]).astype(np.float64)
        np.save(p, out[name]); st.write_text(str(stamp))
    return out

emb = {e: embed_all(e) for e in EMBEDDERS}
results, metric_rows = {}, []
for e in EMBEDDERS:
    d = emb[e]
    res = M.evaluate(d["real"], {n: d[n] for n in SETS}, groups_real=ref_groups, max_dim=MAX_DIM, standardize=True,
                     n_splits=N_SPLITS, n_perm=N_PERM, seed=cfg.seed)
    results[e] = res
    for n in SETS:
        rep = res["report"][n]
        metric_rows.append({"set": n, "embedder": e, "frechet_ratio": rep["w2_sq"]["ratio"],
                            "mean_ratio": rep["mean_sq"]["ratio"], "cov_ratio": rep["cov_sq"]["ratio"],
                            "mmd2": rep["mmd2"]["value"], "mmd_above": rep["mmd2"]["frac_above"],
                            "log_volume": res["spectra"][n].log_volume_ratio})
metrics = pd.DataFrame(metric_rows).pivot(index="set", columns="embedder")
metrics.columns = [f"{m}_{e}" for m, e in metrics.columns]
metrics = metrics.reindex(list(SETS))
metrics[[f"frechet_ratio_{e}" for e in EMBEDDERS]].round(2)

## 6. TSTR (fold 5)

The pilot's classifier and frozen recipe (`tstr.py` holds the identical code). Every set enters
as audio through the A-128 front end. Real-full and Real-matched are trained again here, with the
same code, so every number in the analysis comes from one run. Each condition is saved when it
finishes.

The first Real-full model is kept as the **teacher**: its accuracy on each synthetic set
(train on real, test on synthetic) is one of the exploratory predictors.

In [ ]:
fe128 = C.MelFrontEnd.load(f"{ROOT}/frontend_128mel.json", cfg128, device=device)
def to_mel(w, batch=128):
    return torch.cat([fe128(w[i:i + batch].to(device).float()) for i in range(0, len(w), batch)])

X_test = to_mel(test_wave)
getters = {"Real-full": lambda: T.batch_random_crops(train_split, cfg, fe128, device),
           "Real-matched": lambda: T.batch_fixed(to_mel(matched_wave), matched_y, device)}
getters.update({n: (lambda n=n: T.batch_fixed(to_mel(waves[n]), labels[n], device)) for n in SETS})

tstr_probs = {}
t0 = time.time()
for name, make in getters.items():
    f = OUT_DIR / f"tstr_{name}_{RECIPE['lr']:g}_{RECIPE['steps']}_{int(RECIPE['specaug'])}.npz"
    if f.exists():
        z = np.load(f); tstr_probs[name] = [z[k] for k in sorted(z.files)]
    else:
        get = make(); ps = []
        for s in range(N_SEEDS):
            m = T.train_clf(get, n_classes, seed=s, device=device, **RECIPE)
            ps.append(T.clip_probs(m, X_test, test_clip, n_classes))
            if name == "Real-full" and s == 0:
                torch.save(m.state_dict(), OUT_DIR / "teacher_real_full_seed0.pt")
        np.savez(f, **{f"seed{s}": p for s, p in enumerate(ps)}); tstr_probs[name] = ps
    accs = [T.accuracy(p, test_clip_y) for p in tstr_probs[name]]
    print(f"{name:<14} {np.mean(accs):.3f} +/- {np.std(accs):.3f}   ({(time.time() - t0) / 60:.1f} min so far)")

In [ ]:
teacher = T.ProbeCNN(n_classes).to(device)
teacher.load_state_dict(torch.load(OUT_DIR / "teacher_real_full_seed0.pt", map_location=device)); teacher.eval()
with torch.no_grad():
    trts = {n: float((torch.cat([teacher(x).argmax(1).cpu() for x in to_mel(waves[n]).split(256)]) == labels[n]).float().mean())
            for n in SETS}

acc = {n: [T.accuracy(p, test_clip_y) for p in ps] for n, ps in tstr_probs.items()}
ref = np.mean(acc["Real-matched"])
table = pd.DataFrame({n: {"path": s["path"], "step": s["step"], "guidance": s["guidance"],
                          "tstr": np.mean(acc[n]), "tstr_sd": np.std(acc[n]), "tstr_rel": np.mean(acc[n]) / ref,
                          "trts": trts[n], "learnability": LEARNABILITY[s["path"]]}
                      for n, s in SETS.items()}).T.join(metrics)
for c in table.columns:
    if c != "path":
        table[c] = table[c].astype(float)
print(f"Real-full {np.mean(acc['Real-full']):.3f} +/- {np.std(acc['Real-full']):.3f} | "
      f"Real-matched {ref:.3f} +/- {np.std(acc['Real-matched']):.3f}   (pilot: 0.703, 0.635)")
table[["tstr", "tstr_sd", "tstr_rel", "trts"] + [f"frechet_ratio_{e}" for e in EMBEDDERS]].round(3)

## 7. Primary result: Spearman correlation, bootstrap by path

For each primary predictor: Spearman's rho with relative TSTR accuracy over all sets, and a 95%
interval from resampling the 6 paths with replacement (all sets of a drawn path come along). A
predictor that works gives rho clearly below 0 for the Frechet ratios.

In [ ]:
paths = sorted(table["path"].unique())

def rho(df, col):
    if df[col].nunique() < 2 or df["tstr_rel"].nunique() < 2 or len(df) < 3:
        return np.nan
    return spearmanr(df[col], df["tstr_rel"]).correlation

def path_bootstrap(col, n_boot=N_BOOT, seed=0):
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n_boot):
        pick = rng.choice(paths, len(paths), replace=True)
        vals.append(rho(pd.concat([table[table.path == p] for p in pick]), col))
    vals = np.array(vals); vals = vals[~np.isnan(vals)]
    return np.percentile(vals, [2.5, 97.5]) if len(vals) else (np.nan, np.nan)

PRIMARY = [f"frechet_ratio_{e}" for e in EMBEDDERS]
EXPLORATORY = ([f"{m}_{e}" for e in EMBEDDERS for m in ("mean_ratio", "cov_ratio", "mmd2", "log_volume")]
               + ["trts", "learnability"])
corr_rows = []
for col in PRIMARY + EXPLORATORY:
    lo, hi = path_bootstrap(col)
    corr_rows.append({"predictor": col, "role": "primary" if col in PRIMARY else "exploratory",
                      "spearman_rho": rho(table, col), "ci_lo": lo, "ci_hi": hi, "n_sets": len(table)})
corr = pd.DataFrame(corr_rows).set_index("predictor")
corr.round(3)

In [ ]:
colors = {"A64": "#3b6ea8", "A128": "#1b9e77", "C": "#d95f02", "B_raw": "#7570b3", "B_codec": "#e7298a", "B_latent": "#666666"}
fig, axes = plt.subplots(1, len(EMBEDDERS), figsize=(6.5 * len(EMBEDDERS), 4.6), squeeze=False)
for ax, e in zip(axes[0], EMBEDDERS):
    col = f"frechet_ratio_{e}"
    for n, row in table.iterrows():
        ax.errorbar(row[col], row["tstr_rel"], yerr=row["tstr_sd"] / ref, fmt="none", ecolor="#bbbbbb", lw=1)
        ax.scatter(row[col], row["tstr_rel"], s=60, edgecolors=colors[row["path"]], linewidths=1.5,
                   marker="o" if row["guidance"] == 1.0 else "^",
                   facecolors=colors[row["path"]] if row["step"] == 10000 else "none")
    for p, c in colors.items():
        if p in paths:
            ax.scatter([], [], color=c, label=p)
    ax.scatter([], [], marker="o", color="k", label="guidance 1.0"); ax.scatter([], [], marker="^", color="k", label="guidance 3.0")
    ax.scatter([], [], marker="o", facecolors="none", edgecolors="k", label="5k steps (hollow)")
    ax.set_xscale("log"); ax.axhline(1.0, color="k", lw=0.8, ls=":")
    ax.set_xlabel(f"Frechet / real-vs-real floor ({e}, fold 4)"); ax.set_ylabel("TSTR accuracy / Real-matched (fold 5)")
    rr = corr.loc[col]
    ax.set_title(f"{e}: Spearman {rr.spearman_rho:.2f}  [{rr.ci_lo:.2f}, {rr.ci_hi:.2f}]")
axes[0][0].legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.savefig(OUT_DIR / "metric_vs_tstr.png", dpi=130); plt.show()

## 8. Within factors

* **Across paths** at 10k steps and guidance 1.0: the representation comparison as such.
* **Guidance** (3.0 vs 1.0) and **checkpoint** (10k vs 5k) within each spectrogram path: does the
  metric move in the direction that TSTR moves? "Agrees" means a lower Frechet ratio went with a
  higher TSTR accuracy (or both the other way). A TSTR change smaller than the larger seed sd of
  the two sets is marked as within noise and not counted.

In [ ]:
base = table[(table.step == 10000) & (table.guidance == 1.0)]
print("Across paths (10k, guidance 1.0), n =", len(base))
for col in PRIMARY:
    print(f"   {col}: Spearman {rho(base, col):.2f}")

pair_rows = []
for p in ("A64", "A128", "C"):
    sub = table[table.path == p]
    for factor, a, b in (("guidance", ("guidance", 1.0), ("guidance", 3.0)), ("checkpoint", ("step", 5000), ("step", 10000))):
        fixed = "step" if factor == "guidance" else "guidance"
        for level in sorted(sub[fixed].unique()):
            s = sub[sub[fixed] == level]
            ra, rb = s[s[a[0]] == a[1]], s[s[b[0]] == b[1]]
            if len(ra) != 1 or len(rb) != 1:
                continue
            ra, rb = ra.iloc[0], rb.iloc[0]
            d_tstr = rb.tstr_rel - ra.tstr_rel
            noise = max(ra.tstr_sd, rb.tstr_sd) / ref
            row = {"path": p, "factor": factor, f"at {fixed}": level, "delta_tstr_rel": d_tstr, "noise": noise}
            for col in PRIMARY:
                d_m = rb[col] - ra[col]
                row[f"delta_{col}"] = d_m
                row[f"agrees_{col}"] = ("within noise" if abs(d_tstr) <= noise else ("yes" if np.sign(d_m) == -np.sign(d_tstr) else "no"))
            pair_rows.append(row)
pairs = pd.DataFrame(pair_rows)
if pairs.empty:
    print("no complete guidance or checkpoint pairs among the sets present")
else:
    display(pairs.round(3))
for col in (PRIMARY if not pairs.empty else []):
    v = pairs[f"agrees_{col}"]
    print(f"{col}: agrees {int((v == 'yes').sum())}, disagrees {int((v == 'no').sum())}, within noise {int((v == 'within noise').sum())}")

## 9. Save

In [ ]:
out = {"commit": commit, "quick_test": QUICK_TEST, "recipe": RECIPE, "pilot_file": Path(pilots[-1]).name,
       "real_full": acc["Real-full"], "real_matched": acc["Real-matched"],
       "table": table.reset_index().rename(columns={"index": "set"}).round(5).to_dict("records"),
       "correlations": corr.reset_index().round(4).to_dict("records"),
       "within_factor": pairs.round(5).to_dict("records"),
       "missing_sets": missing,
       "reports": {e: r["report"] for e, r in results.items()}}
with open(OUT_DIR / f"study_{commit}.json", "w") as f:
    json.dump(out, f, indent=2, default=float)
print("saved", OUT_DIR / f"study_{commit}.json")

## How to read the result

* **The primary result is the two Spearman correlations with their path-bootstrap intervals.**
  An interval entirely below 0: the metric predicts usefulness across representations. An interval
  that includes 0: it does not, at this resolution. Either is a result for the thesis; the design
  was fixed so that neither can be argued away afterwards.
* **Six paths make the interval wide.** Resampling paths is the honest unit (sets of one path share
  a model), so a correlation that looks strong over 15 points can still have a wide interval.
* **Within-factor agreement** says whether the metric is useful for the decisions a practitioner
  actually makes: which checkpoint, which guidance.
* **Exploratory predictors** are reported for completeness; none of them can be promoted to a
  primary result after the fact.